# Model implementation on tunder

In this notebook, we presented how we adapt the continous integration (MS2) to work on tunder machine.

<a id="table"></a>

## Table of Contents

- [How to use the package](#setup)

    - [Set the envrioment using pip or conda](#envrioment)
    - [Computing capacity required](#power)
    - [Data location](#data)
    - [Bash script to process all the analysis](#bash)


- [Processing explanation](#model_explanation)

    - [Satellite images download](#satellite)
    - [Transposing the dataset and adding date](#transpose)
    - [Merging historical with newly acquired data](#merge)
    - [Analysis](#analysis)
    - [Tiff creation](#tiff)



<a id="setup"></a>

[Go back to the table](#table)

## How to use the package

Here we will describe how to setup the envrioment unsing pip or conda, the computing capacity used and where the data are located

<a id="envrioment"></a>

### Setup the envtioment

The envrioment is already avaible in the folder, it is activated at launch everytime

It possible to setup the envrioment using Conda or Pip. the following command will create the envrioment based on the preferred choice.

Using pip

python3 -m venv ndvi
source ndvi/bin/activate
pip install -r requirements.txt

Or using conda

conda env create -f environment.yml --name ndvi


<a id="power"></a>

### Computing capacity required

To parallelize the computation, we use the package Dask. Each script will use different power capacities according to the computing power needed. The highest computing capacity used is in script X, using Y wokrer and Z total RAM

<a id="data"></a>

### Data location

The data stored in the demo are just a few pixel to test the script. The full data are deposit in the following path:

/mnt/data1/UniBe-swiss-ndvi/data/

Here the lookuptable, the historical data to process, the final TIFF and the temporary data on the continous setup

<a id="bash"></a>


### Bash script to process all the analysis

We created a bash script that automatically launch all the script in sequence. The script will perform the following action:

- retrieve the last date analysed by the model
- set the current date as the last date to analyse
- exectue the script 1 to search and download the new satellite images
- if no satellite images are detected, it skip all the computation and overwrite the last date to the current date
- if one (or more) satellite images are found, exectue in order all the script from 2 to 6
- after the computation are finished, it overwrites the last date to the current date

We introduce this dynamic windows of last date analysed and current date to have a system that is flexible (can be aither run once per day or with some days lag).

The window date is automatically set from the most recent day not processed yet to the current date at which the bash is activated. 

In case of 



<a id="model_explanation"></a>

[Go back to the table](#table)

## Processing explanation

Here we will describe in detail all the script present in the demo, how they work and the output of each script.


<a id="satellite"></a>

### Satellite images download

The script workflow_implementation/demo/test_all_pixels/1_extract_swisstopo_dataset.py. The script uses stac to access the data stored on 'https://data.geo.admin.ch/api/stac/v0.9/'.


<a id="transpose"></a>

[Go back to the table](#table)

### Transposing the dataset and adding date

TODO

<a id="merge"></a>


### Merging historical with newly acquired data

The merging of previously analysed data (historical) and the new acquired data.

The zarr dataset will have the following information:

- Data

    - NDVI timeserie (pixel X time)
    - Median NDVI obtained from Samantha model (pixel X time)
    - Mask specifying the information of NDVI value (pixel X time)
    - Boolean array specifying if a date has some observation or not (time)

- Coordinates

    - pixel
    - date
    - X and Y position on the dataset
    - X and Y coordinates used to create the TIFF for each time layer

The folder have a size of TODO

<a id="analysis"></a>

### Analysis

The analysis performed on tunder uses the function described in TODO

Here there are the summarised step of the function, which is applied for each pixel indipendentely.

To smooth a value is needed a 7 windows of observation and the value of interest must be at the center, we load the timeseries from the last third observation up to the current date.

- Step 1: load the data from the last third observation up to the current date

- Step 2: filter the data, including only the observation data (the data must be an observation and have a value between 0 and 1)

- Step 3: perform the outlier detection based on 

    - the difference between the NDVI and corresponded median value

    - the difference between the current delta (defined as the difference between the NDVI and corresponded median value and the neighoubring deltas)

- If both conditions are met, the data is an outlier and is removed from the analysis

- Step 4: the smoothing is performed on the NDVI value wichi do not present extreme negative events or values close to the boundaries condition:

- For each date in the remaining timeserie, we evaluate a rolling window of 7 values:

    - if at least 5 out 7 values have a delta of -0.2 (extreme negative NDVI) or the NDVI values is above 0.95 or below 0.05, we skip the smoothing 

    - if the case above does not occour, the smoothing of the fourth value (in the middle is performed)

- After the smoothing the delta smoothed and not smoothed are collected to linearly interpolate the deltas to the full timeserie and summed to the median NDVI to obtained the processed NDVI values

- The mask NDVI array is created based on ...

- The data are wrote back to be used for the TIFF generation and re-used for the merging of the future data

TODO: benchmark

<a id="tiff"></a>


### Tiff creation

TODO
